# Session 15: Distributions, Outliers and the Titanic

Data Science with Python, Level 1, Stage 2: Working with Data

## Learning Objectives

By the end of this session you will be able to:

1. Describe centre, spread and shape of a distribution.
2. Find outliers with the IQR rule and z-scores.
3. Decide what to do with an outlier.
4. Turn a big question into small answerable ones.
5. Compare survival rates across groups.
6. Tell a data story with honest caveats.

## Where We Are

In Session 14 the box plots showed dots beyond the whiskers. Those are **outliers**: values far away from the rest. Some are mistakes, such as an age of 230. Others are real but rare, such as a luxury flat. In the first half you will describe distributions with numbers and find outliers with two standard rules, using 800 practice apartments.

In the second half you put the whole of Stage 2 together in a **case study** on real data: the passenger list of the Titanic. You will turn one big question, who survived and why, into small questions that pandas and charts can answer, and finish with an honest story. This is exactly what Project 2 asks you to do next session.

## Setup

Run this cell first. It loads the libraries and sets the address of the course datasets.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"
flats = pd.read_csv(DATA_URL + "apartments.csv")

titanic = sns.load_dataset("titanic")

## 1. Describing a Distribution

In [ ]:
sns.histplot(data=flats, x="price_k", bins=40)
plt.title("Apartment prices (thousands)")
plt.show()

In [ ]:
price = flats["price_k"]
print("Mean:  ", round(price.mean(), 1))
print("Median:", price.median())
print("Std:   ", round(price.std(), 1))
print("Skew:  ", round(price.skew(), 2))

## 2. Quantiles and the IQR Rule

A **quantile** splits the sorted values: the 0.25 quantile, called the first quartile Q1, has a quarter of the values below it. The median is the 0.5 quantile. The **interquartile range** (IQR) is Q3 minus Q1, the width of the middle half of the data.

The **IQR rule**, used by every box plot, calls a value an outlier when it lies more than 1.5 IQR below Q1 or above Q3.

In [ ]:
q1, q3 = price.quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr

print(f"Q1 {q1:.1f}, Q3 {q3:.1f}, IQR {iqr:.1f}")
print(f"Fences: {low:.1f} to {high:.1f}")
outliers = flats[(price < low) | (price > high)]
print(len(outliers), "outliers")

## 3. z-Scores

A **z-score** measures how far a value is from the mean, in standard deviations: `z = (x - mean) / std`. A z-score of 2 means two standard deviations above the mean. A common rule calls a value an outlier when its z-score is above 3 or below -3. This works best for distributions that are roughly symmetric.

In [ ]:
flats["price_z"] = (price - price.mean()) / price.std()

extreme = flats[flats["price_z"].abs() > 3]
print(len(extreme), "flats with |z| > 3")
extreme[["district", "area_m2", "price_k", "price_z"]].round(2)

## 4. Context Matters: Outliers per Group

In [ ]:
flats["price_per_m2"] = flats["price_k"] * 1000 / flats["area_m2"]

order = flats.groupby("district")["price_per_m2"].median().sort_values().index
sns.boxplot(data=flats, x="district", y="price_per_m2", order=order)
plt.title("Price per square metre by district")
plt.show()

## 5. What to Do with Outliers

| Situation | Action | Code |
| --- | --- | --- |
| Impossible value (age 230) | Fix it, or set to missing | `df.loc[mask, col] = np.nan` |
| Real but extreme, and it matters | Keep it; use the median and robust charts | `df[col].median()` |
| Real, but it distorts a model | Cap it at a limit (winsorise) | `df[col].clip(upper=high)` |
| Outside the question you study | Remove it, and say so | `df[~mask]` |

Never delete outliers just to make the data look nicer. Write down every decision and its reason.

In [ ]:
capped = price.clip(upper=high)
print("Mean before:", round(price.mean(), 1), "after capping:", round(capped.mean(), 1))
print("Median before:", price.median(), "after capping:", capped.median())

## 6. Log Scales for Skewed Data

In [ ]:
flats["log_price"] = np.log(flats["price_k"])
print("Skew of price:", round(flats["price_k"].skew(), 2))
print("Skew of log price:", round(flats["log_price"].skew(), 2))

sns.histplot(data=flats, x="log_price", bins=40)
plt.title("Log of apartment prices")
plt.show()

## 7. Questions and Data

In [ ]:
print(titanic.shape)
titanic.head()

| Column | Meaning | Example |
| --- | --- | --- |
| `survived` | 1 if the passenger survived, else 0 | `1` |
| `pclass`, `class` | Ticket class, 1 is the most expensive | `3`, `Third` |
| `sex`, `age` | Sex and age in years | `female`, `38.0` |
| `sibsp`, `parch` | Siblings or spouses, parents or children aboard | `1`, `0` |
| `fare` | Ticket price in pounds of 1912 | `71.28` |
| `embarked` | Port of boarding: S, C or Q | `C` |
| `who`, `alone` | man, woman or child; travelling alone | `child`, `True` |

## 8. Data Quality

In [ ]:
titanic.isna().sum().sort_values(ascending=False).head(5)

## 9. Survival Overall and by Group

In [ ]:
print(round(titanic["survived"].mean(), 3))
print(titanic.groupby("sex")["survived"].mean().round(3))

In [ ]:
titanic.groupby("class")["survived"].agg(["count", "mean"]).round(3)

## 10. Combining Factors

In [ ]:
pd.crosstab(titanic["class"], titanic["sex"],
            values=titanic["survived"], aggfunc="mean").round(3)

## 11. New Columns for New Questions

In [ ]:
titanic.groupby("class")["fare"].median()

## 12. Telling the Story

## Common Mistakes

| Symptom | Mistake | Fix |
| --- | --- | --- |
| Biased summary | Reporting the mean of strongly skewed data | Report the median, or both. |
| Data destroyed | Deleting every value the IQR rule flags | Look at each outlier and decide with reasons. |
| Misleading rate | Comparing counts of survivors instead of rates | Compare means of `survived`, because group sizes differ. |
| Overclaiming | Writing that class caused survival | Describe patterns; offer explanations as possibilities. |

## In-Class Exercises

Work through these exercises during the session. Write your answer in the code cell under each one.

### In-Class Exercise 1: Area Distribution

Draw a histogram of `area_m2` and print its mean, median and skew. Is it skewed like the prices?

In [ ]:
# Write your code below

### In-Class Exercise 2: Cap and Compare

Cap the rentals in `bike_rentals.csv` at their 1st and 99th percentiles with `clip(lower=..., upper=...)`, and compare the mean and standard deviation before and after.

In [ ]:
# Write your code below

### In-Class Exercise 3: Children by Class

Using `who`, compare the survival rate of children in each class. Was being a child enough in third class?

In [ ]:
# Write your code below

### In-Class Exercise 4: Family Size

Create `family_size` and plot the survival rate for each family size with a bar plot. Is bigger always better?

In [ ]:
# Write your code below

## Summary

- The median resists outliers; the mean is pulled towards the tail.
- The IQR rule: below Q1 - 1.5 IQR or above Q3 + 1.5 IQR.
- A z-score counts standard deviations from the mean.
- Investigate outliers before you remove, cap or keep them.
- Break a big question into small ones that data can answer.
- A good story states findings with numbers and honest caveats.

## What's Next

Next session is Project 2. You will choose a dataset, ask four questions, and answer them with numbers and charts in a report that someone else can read without you.